# Prompt format vs. token usage: exploratory analysis

**Research question:** Does prompt format (structured vs. informal) affect token usage when an LLM writes Python code for the same requirements?

- **Independent variable:** `format`, either `structured` (labeled sections, bulleted requirements) or `informal` (one casual paragraph)
- **Dependent variables:** `input_tokens`, `output_tokens` (from the API's `usage` field)
- **Secondary check:** hidden unit tests passed (one test per requirement)
- **Controls:** same model, temperature, `max_tokens`, and system prompt on every call; no prompt caching; no extended thinking

**Unit of analysis.** Each task has 5 runs per format, but those runs are repeated measurements of the *same* prompt, not independent samples. Statistical tests therefore compare **per-task means** (10 pairs). Treating all 100 rows as independent would overstate certainty (pseudo-replication).

> The "How to read it" notes below explain each plot. Lines marked **Result:** are placeholders to fill in after the real run.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
RESULTS_CSV = "../data/results.csv"
MODEL = "claude-haiku-4-5"   # analyze one model at a time

FORMATS = ["structured", "informal"]
COLORS = {"structured": "#2a78d6", "informal": "#eb6834"}  # blue / orange (colorblind-safe pair)
NEUTRAL = "#8a8984"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae",
    "axes.grid": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "font.size": 10,
})

df = pd.read_csv(RESULTS_CSV)
df = df[df["model"] == MODEL].copy()
df["pass_rate"] = df["tests_passed"] / df["tests_total"]
print(f"{len(df)} rows for {MODEL}")
df.head()

## 1. Sanity checks

In [ ]:
print("Runs per task x format (every cell should equal N_RUNS):")
display(df.pivot_table(index="task_id", columns="format", values="run", aggfunc="count"))

print("stop_reason counts ('max_tokens' would mean a truncated answer):")
display(df["stop_reason"].value_counts())

print("Missing values per column:")
display(df.isna().sum()[lambda s: s > 0] if df.isna().any().any() else "none")

**How to read it:** every task × format cell should show the same count (N_RUNS). Any `max_tokens` stop reason means a response was cut off, and its `output_tokens` is capped, not natural. Investigate those rows before trusting the token comparisons.

**Result:** _fill in after the run._

## 2. Summary table: tokens by format

In [ ]:
summary = df.groupby("format")[["input_tokens", "output_tokens"]].agg(["mean", "median", "std"]).round(1)
summary.loc[FORMATS]

**How to read it:** this table pools all tasks and runs, so it's a first look and not the test. Compare the means and medians between rows. A big gap between mean and median suggests a few long responses are pulling the mean.

**Result:** _fill in after the run._

## 3. Distribution of output tokens by format

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
data = [df.loc[df["format"] == f, "output_tokens"] for f in FORMATS]
ax.boxplot(data, tick_labels=FORMATS, widths=0.45, showfliers=False,
           medianprops={"color": "#0b0b0b", "linewidth": 2},
           boxprops={"color": NEUTRAL}, whiskerprops={"color": NEUTRAL}, capprops={"color": NEUTRAL})
rng = np.random.default_rng(SEED)
for i, (f, values) in enumerate(zip(FORMATS, data), start=1):
    jitter = rng.uniform(-0.12, 0.12, size=len(values))
    ax.scatter(i + jitter, values, s=18, color=COLORS[f], alpha=0.7, edgecolor="white", linewidth=0.5)
ax.set_ylabel("output tokens")
ax.set_title("Output tokens per response, by prompt format", loc="left")
ax.grid(axis="x", visible=False)
plt.show()

**How to read it:** the box spans the middle 50% of responses and the black line is the median. Each dot is one response. Shifted or differently sized boxes suggest format matters, but tasks differ a lot in how much code they need, so this pooled view mixes task difficulty with format. Section 4 controls for that.

**Result:** _fill in after the run._

## 4. Paired comparison per task (slope plot)

In [ ]:
task_means = df.groupby(["task_id", "format"])[["input_tokens", "output_tokens", "pass_rate"]].mean().unstack()
out = task_means["output_tokens"][FORMATS]

fig, ax = plt.subplots(figsize=(6, 5))
for task_id, row in out.iterrows():
    ax.plot([0, 1], [row["structured"], row["informal"]], color=NEUTRAL, linewidth=1.5, zorder=1)
for x, f in enumerate(FORMATS):
    ax.scatter([x] * len(out), out[f], s=50, color=COLORS[f], edgecolor="white", linewidth=1.5, zorder=2)

# Task labels at the right end, nudged apart so close values don't overlap.
min_gap = (out.values.max() - out.values.min()) * 0.045
label_y = None
for task_id, y in out["informal"].sort_values().items():
    label_y = y if label_y is None else max(y, label_y + min_gap)
    ax.text(1.06, label_y, task_id, va="center", fontsize=8, color="#52514e")
ax.set_ylim(top=max(ax.get_ylim()[1], label_y + min_gap))  # keep the top label inside the plot

ax.set_xticks([0, 1], FORMATS)  # the x-axis names the formats, so no legend is needed
ax.set_xlim(-0.2, 1.5)
ax.set_ylabel("mean output tokens (over runs)")
ax.set_title("Each line is one task: structured → informal", loc="left", pad=12)
ax.grid(axis="x", visible=False)
plt.show()

print(f"Tasks where informal used MORE output tokens: {(out['informal'] > out['structured']).sum()} of {len(out)}")

**How to read it:** each line connects the same task under both formats, so differences in task difficulty cancel out. If most lines slope the same way, format has a consistent effect. If they cross in both directions, the effect depends on the task or is noise.

**Result:** _fill in after the run._

## 5. Input-token gap per task (confound check)

In [ ]:
inp = task_means["input_tokens"][FORMATS]
input_gap = (inp["structured"] - inp["informal"]).sort_values()
output_gap = (out["structured"] - out["informal"]).loc[input_gap.index]

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(input_gap.index, input_gap.values, color=NEUTRAL, height=0.6)
ax.axvline(0, color="#52514e", linewidth=1)
ax.set_xlabel("input tokens: structured − informal")
ax.set_title("Prompt length gap per task", loc="left")
ax.grid(axis="y", visible=False)
plt.show()

rho, p = stats.spearmanr(input_gap, output_gap)
print(f"Spearman correlation between input gap and output gap across tasks: rho={rho:.2f}, p={p:.3f}")

**How to read it:** bars to the right mean the structured prompt was longer for that task. Prompt length is a possible **confound**: if longer prompts lead to longer answers, an output difference could come from length rather than format. A strong correlation between the input gap and the output gap would point that way. With only 10 tasks, treat this as a warning sign, not proof.

**Result:** _fill in after the run._

## 6. Run-to-run variability

In [ ]:
var = df.groupby(["task_id", "format"])["output_tokens"].agg(["mean", "std"])
var["cv"] = var["std"] / var["mean"]
cv = var["cv"].unstack()[FORMATS]

fig, ax = plt.subplots(figsize=(6, 4.5))
y = np.arange(len(cv))
ax.hlines(y, cv.min(axis=1), cv.max(axis=1), color="#d8d7d1", linewidth=2, zorder=1)
for f in FORMATS:
    ax.scatter(cv[f], y, s=50, color=COLORS[f], edgecolor="white", linewidth=1.5, zorder=2, label=f)
ax.set_yticks(y, cv.index)
ax.set_xlabel("coefficient of variation of output tokens (std / mean)")
ax.set_title("How much output length varies across the runs of one prompt", loc="left")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="lower right")
plt.show()

var.round(2).unstack()

**How to read it:** the coefficient of variation (CV) is the standard deviation divided by the mean, so tasks with different lengths can be compared. A CV of 0.10 means runs typically differ by about 10% from their average. If one format is consistently more variable, its prompts leave the model more room to choose how much to write. High variability overall also means more runs would be needed to detect small format effects.

**Result:** _fill in after the run._

## 7. Output tokens vs. correctness

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
rng = np.random.default_rng(SEED)
for f in FORMATS:
    sub = df[df["format"] == f]
    jitter = rng.uniform(-0.02, 0.02, size=len(sub))
    ax.scatter(sub["output_tokens"], sub["pass_rate"] + jitter, s=24, color=COLORS[f],
               alpha=0.7, edgecolor="white", linewidth=0.5, label=f)
ax.set_xlabel("output tokens")
ax.set_ylabel("fraction of tests passed")
ax.set_ylim(-0.08, 1.08)
ax.set_title("Does writing more mean passing more tests?", loc="left")
ax.legend(frameon=False, loc="lower right")
plt.show()

**How to read it:** each dot is one response, with a small vertical jitter so overlapping dots stay visible. If correct answers (top row) cluster at a certain length, length and correctness are linked. If one format's dots sit lower, that format produced more failing code, and a token "saving" might just come from missing requirements.

**Result:** _fill in after the run._

## 8. Output tokens per fully correct solution

In [ ]:
eff = df.groupby("format").agg(
    responses=("output_tokens", "size"),
    total_output_tokens=("output_tokens", "sum"),
    fully_correct=("passed_all", "sum"),
    mean_pass_rate=("pass_rate", "mean"),
).loc[FORMATS]
eff["tokens_per_correct_solution"] = (eff["total_output_tokens"] / eff["fully_correct"]).round(1)
eff["mean_pass_rate"] = eff["mean_pass_rate"].round(3)
eff

**How to read it:** `tokens_per_correct_solution` is the total output tokens spent divided by the number of responses that passed every test. It combines cost and quality: a format that writes fewer tokens but fails more often can still be *less* efficient by this measure.

**Result:** _fill in after the run._

## 9. Paired statistical comparison (within task)

In [ ]:
def paired_comparison(metric, n_boot=10_000):
    """Compare informal vs structured using per-task means (one pair per task)."""
    means = task_means[metric][FORMATS]
    diff = means["informal"] - means["structured"]

    # Wilcoxon signed-rank: are the differences centered on 0? (Fails if every diff is 0.)
    try:
        w_stat, w_p = stats.wilcoxon(diff)
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    # Bootstrap: resample the tasks with replacement and recompute the mean difference.
    rng = np.random.default_rng(SEED)
    samples = rng.choice(diff.to_numpy(), size=(n_boot, len(diff)), replace=True)
    boot_means = samples.mean(axis=1)
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

    return {
        "metric": metric,
        "n_tasks": len(diff),
        "mean_diff (informal - structured)": round(diff.mean(), 3),
        "median_diff": round(diff.median(), 3),
        "bootstrap 95% CI": f"[{ci_low:.3f}, {ci_high:.3f}]",
        "wilcoxon_stat": w_stat,
        "wilcoxon_p": round(w_p, 4) if not np.isnan(w_p) else np.nan,
    }

pd.DataFrame([paired_comparison(m) for m in ["output_tokens", "input_tokens", "pass_rate"]]).set_index("metric")

**How to read it:**
- **mean_diff** is positive when informal prompts used more tokens (or passed more tests) than structured ones, averaged over tasks.
- **Bootstrap 95% CI:** we resample the 10 tasks 10,000 times and take the middle 95% of the resampled mean differences. If the interval excludes 0, the difference is unlikely to be just task-to-task noise.
- **Wilcoxon signed-rank p-value:** a non-parametric paired test. It asks whether the per-task differences are consistently on one side of zero, without assuming they are normally distributed.
- **Power caveat:** with only 10 pairs the test can detect only fairly consistent effects; the smallest possible two-sided p-value is about 0.002 (all 10 differences in the same direction). A non-significant result means "no evidence of a difference at this sample size", not "no difference".

**Result:** _fill in after the run._

## 10. Conclusions and limitations

**Conclusions:** _fill in after the run: direction and size of the output-token difference, whether the CI excludes 0, whether correctness differs, and whether the input-length gap could explain it._

**Limitations to discuss:**
- **One model** (`claude-haiku-4-5`) at one temperature; results may not carry over to other models or to temperature 0.
- **One prompt pair per task:** the specific wording of each informal paragraph is mixed in with "informality" itself. More paraphrases per format would separate the two.
- **Small sample:** 10 tasks means low statistical power and wide confidence intervals.
- **Prompt length differs between formats** (section 5), which is a possible confound.
- **Small, self-contained tasks:** real-world coding prompts are longer and messier.